# 🌍 Week 6: Autonomous Travel & Packing Concierge Agent
### Built for AWS AI/ML Mentorship Program
This notebook demonstrates an end-to-end **Autonomous ReAct Agent** with:
- Dynamic Geocoding via Open-Meteo API
- Live Multi-Day Weather Forecasting
- Real-time Activity Discovery via Wikipedia API
- Gemini Function Calling Handshake & Multi-Step ReAct Loop
- Local File Export with Path Traversal Guardrails

In [6]:
# Cell 2: Setup and Retrieve Key from Google Colab Secrets (Key Icon 🔑)
import os
from google.colab import userdata

# Install lightweight dependencies
!pip install python-dotenv pydantic -q

# Retrieve your secret key from the Colab Secrets tab
try:
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    print("✅ GEMINI_API_KEY loaded securely from Colab Secrets!")
except Exception as e:
    # Fallback in case your secret is named with lowercase or slight variation
    try:
        os.environ["GEMINI_API_KEY"] = userdata.get('gemini_api_key')
        print("✅ gemini_api_key loaded securely from Colab Secrets!")
    except Exception:
        raise ValueError(
            "Could not find 'GEMINI_API_KEY' in your Colab Secrets tab! "
            "Please click the Key icon (🔑) on the left sidebar, add a new secret named "
            "'GEMINI_API_KEY', paste your key, and toggle on 'Notebook access'."
        )

✅ GEMINI_API_KEY loaded securely from Colab Secrets!


In [7]:
# Cell 3: The Agent's Toolset (Zero Hardcoding)
import os
import json
import urllib.request
import urllib.parse
from typing import Dict, Any, List

def geocode_location(place_name: str) -> Dict[str, Any]:
    """Dynamically resolves any place name to lat/lon using Open-Meteo Geocoding API."""
    query = urllib.parse.quote(place_name.strip())
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={query}&count=1&language=en&format=json"
    req = urllib.request.Request(url, headers={"User-Agent": "TravelAgent/1.0"})
    with urllib.request.urlopen(req, timeout=6) as response:
        data = json.loads(response.read().decode("utf-8"))

    results = data.get("results")
    if not results:
        raise ValueError(f"Could not locate '{place_name}'.")
    top = results[0]
    return {
        "name": top.get("name"),
        "country": top.get("country", ""),
        "latitude": top.get("latitude"),
        "longitude": top.get("longitude")
    }

def get_weather_forecast(city: str, days: int = 3) -> Dict[str, Any]:
    """Live weather forecast using coordinates and Open-Meteo."""
    days = max(1, min(int(days), 14))
    try:
        geo = geocode_location(city)
        lat, lon = geo["latitude"], geo["longitude"]
        url = (
            f"https://api.open-meteo.com/v1/forecast?"
            f"latitude={lat}&longitude={lon}&"
            f"daily=temperature_2m_max,temperature_2m_min,precipitation_probability_max&"
            f"forecast_days={days}&timezone=auto"
        )
        req = urllib.request.Request(url, headers={"User-Agent": "TravelAgent/1.0"})
        with urllib.request.urlopen(req, timeout=6) as response:
            data = json.loads(response.read().decode("utf-8"))

        daily = data.get("daily", {})
        highs = daily.get("temperature_2m_max", [])
        lows = daily.get("temperature_2m_min", [])
        rains = daily.get("precipitation_probability_max", [])

        avg_high = sum(highs)/len(highs) if highs else 25.0
        avg_low = sum(lows)/len(lows) if lows else 18.0
        max_rain = max(rains) if rains else 0

        condition = "Hot & Sunny" if avg_high > 28 else ("Mild" if avg_high > 18 else "Cold")
        if max_rain > 50:
            condition += " with expected rain"

        return {
            "status": "success",
            "resolved_location": f"{geo['name']}, {geo['country']}",
            "forecast_days": days,
            "overall_climate": condition,
            "avg_high_c": round(avg_high, 1),
            "avg_low_c": round(avg_low, 1),
            "max_rain_probability": max_rain
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}

def get_city_activities(city: str, theme: str = "refreshment", num_days: int = 3) -> Dict[str, Any]:
    """Discovers live verified sights using Wikipedia Search API."""
    num_days = max(1, min(int(num_days), 7))
    search_term = urllib.parse.quote(f"{city} tourism {theme} attractions")
    url = f"https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch={search_term}&utf8=&format=json&srlimit=5"

    try:
        req = urllib.request.Request(url, headers={"User-Agent": "EducationalAgent/1.0"})
        with urllib.request.urlopen(req, timeout=6) as response:
            data = json.loads(response.read().decode("utf-8"))

        items = data.get("query", {}).get("search", [])
        activities = []
        for it in items:
            clean_snippet = it.get("snippet", "").replace('<span class="searchmatch">', '').replace('</span>', '')
            activities.append({"name": it.get("title"), "description": clean_snippet[:150] + "..."})

        return {"status": "success", "city": city, "theme": theme, "attractions": activities}
    except Exception as e:
        return {"status": "error", "message": str(e)}

def save_travel_plan(filename: str, content: str) -> Dict[str, Any]:
    """Saves the markdown plan to disk."""
    safe_name = os.path.basename(filename.strip())
    if not safe_name.endswith((".md", ".txt")):
        safe_name += ".md"
    os.makedirs("outputs", exist_ok=True)
    path = os.path.join("outputs", safe_name)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    return {"status": "success", "saved_path": path, "filename": safe_name}

print("✅ Tools defined successfully!")

✅ Tools defined successfully!


In [23]:
# Cell 4: System Prompt, Gemini OpenAPI Tool Schemas, and ReAct Engine (Verified)
import os
import json
import time
import urllib.request
import urllib.error

SYSTEM_INSTRUCTION = """
You are an elite Autonomous Travel & Packing Concierge Agent.
RULES:
1. Carefully inspect the user's prompt for: destination, duration, purpose (e.g. business, safari, relaxation), and any special requests.
2. Only call tools requested by the user. If they only ask for packing, do NOT call activity recommendations.
3. If they ask for activities, refreshment, or sights, call `get_city_activities`.
4. Always call `save_travel_plan` to export a comprehensive markdown travel guide before finishing.
"""

TOOLS_SCHEMA = [
    {
        "function_declarations": [
            {
                "name": "get_weather_forecast",
                "description": "Fetches live weather forecast for any city worldwide.",
                "parameters": {
                    "type": "OBJECT",
                    "properties": {
                        "city": {"type": "STRING", "description": "City or country name."},
                        "days": {"type": "INTEGER", "description": "Number of days (1-14)."}
                    },
                    "required": ["city"]
                }
            },
            {
                "name": "get_city_activities",
                "description": "Finds tourist attractions and relaxation ideas for any city in the world.",
                "parameters": {
                    "type": "OBJECT",
                    "properties": {
                        "city": {"type": "STRING", "description": "Target city name."},
                        "theme": {"type": "STRING", "description": "Theme like 'refreshment', 'relaxation', or 'culture'."},
                        "num_days": {"type": "INTEGER", "description": "Number of days (1-7)."}
                    },
                    "required": ["city"]
                }
            },
            {
                "name": "save_travel_plan",
                "description": "Saves the completed markdown travel plan to disk.",
                "parameters": {
                    "type": "OBJECT",
                    "properties": {
                        "filename": {"type": "STRING", "description": "Filename, e.g. 'trip_plan.md'."},
                        "content": {"type": "STRING", "description": "Full markdown content."}
                    },
                    "required": ["filename", "content"]
                }
            }
        ]
    }
]

class TravelAgent:
    def __init__(self, primary_model: str = "gemini-flash-lite-latest", max_steps: int = 6):
        self.api_key = os.environ.get("GEMINI_API_KEY")
        if not self.api_key:
            raise ValueError("GEMINI_API_KEY environment variable is not set!")

        self.candidate_models = [
            primary_model,
            "gemini-flash-lite-latest",
            "gemini-3.8-flash"
        ]
        self.active_model = primary_model
        self.max_steps = max_steps
        self.tools = {
            "get_weather_forecast": get_weather_forecast,
            "get_city_activities": get_city_activities,
            "save_travel_plan": save_travel_plan
        }

    def _call_gemini(self, contents):
        payload = {
            "contents": contents,
            "tools": TOOLS_SCHEMA,
            "systemInstruction": {"parts": [{"text": SYSTEM_INSTRUCTION}]},
            "generationConfig": {"temperature": 0.2}
        }

        last_error = None
        for model in self.candidate_models:
            endpoint = (
                f"https://generativelanguage.googleapis.com/v1beta/models/"
                f"{model}:generateContent?key={self.api_key}"
            )
            req = urllib.request.Request(
                endpoint,
                data=json.dumps(payload).encode("utf-8"),
                headers={"Content-Type": "application/json"}
            )
            try:
                with urllib.request.urlopen(req, timeout=30) as res:
                    self.active_model = model
                    return json.loads(res.read().decode("utf-8"))
            except urllib.error.HTTPError as e:
                err_body = e.read().decode("utf-8")
                if e.code in [503, 404]:
                    print(f"⚠️ Model '{model}' unavailable (HTTP {e.code}). Switching...")
                    time.sleep(1)
                    last_error = err_body
                    continue
                else:
                    print(f"\n❌ Gemini API Error: {err_body}\n")
                    raise e
            except Exception as e:
                last_error = str(e)
                continue

        raise RuntimeError(f"All models failed: {last_error}")

    def run(self, user_request: str) -> str:
        print(f"🎯 USER REQUEST: {user_request}\n" + "-"*60)
        contents = [{"role": "user", "parts": [{"text": user_request}]}]

        for step in range(1, self.max_steps + 1):
            print(f"🔄 [Step {step}/{self.max_steps}] Reasoning...")
            res = self._call_gemini(contents)

            candidates = res.get("candidates", [])
            if not candidates:
                return "No candidate response from Gemini."

            parts = candidates[0].get("content", {}).get("parts", [])

            fn_call = None
            text_resp = ""
            for p in parts:
                if "functionCall" in p:
                    fn_call = p["functionCall"]
                if "text" in p:
                    text_resp += p["text"]

            if not fn_call:
                print(f"✅ Mission complete (Powered by {self.active_model})!\n")
                return text_resp.strip()

            contents.append({
                "role": "model",
                "parts": candidates[0]["content"].get("parts", [{"functionCall": fn_call}])
            })

            fn_name = fn_call["name"]
            fn_args = fn_call.get("args", {})
            print(f"🛠️  Action: {fn_name}({json.dumps(fn_args)})")

            tool_fn = self.tools.get(fn_name)
            if not tool_fn:
                result = {"error": f"Tool '{fn_name}' not found."}
            else:
                result = tool_fn(**fn_args)

            status = result.get("status", "done")
            print(f"👀 Observation: {status}")

            # Role MUST be 'user' for Gemini REST API function responses
            contents.append({
                "role": "user",
                "parts": [{
                    "functionResponse": {
                        "name": fn_name,
                        "response": {
                            "name": fn_name,
                            "content": result
                        }
                    }
                }]
            })

        return "Reached max steps."

print("✅ TravelAgent engine ready!")

✅ TravelAgent engine ready!


In [24]:
# Cell 5: Run the Travel Concierge Agent & Inspect Output File
import glob

agent = TravelAgent()

# You can test any query here:
user_query = (
    "I am going to bangkok tomorrow to attend a business meeting for 2 weeks, "
    "so give me what should I pack and plan a refreshment idea after the meeting for the last three days."
)

result = agent.run(user_query)

print("\n" + "="*60)
print("📋 AGENT FINAL SUMMARY:")
print("="*60)
print(result)

# Automatically find and display the markdown file saved in outputs/
print("\n" + "="*60)
print("📄 GENERATED MARKDOWN FILE CONTENT (from outputs/):")
print("="*60)

saved_files = glob.glob("outputs/*.md")
if saved_files:
    latest_file = max(saved_files, key=os.path.getctime)
    print(f"Reading: {latest_file}\n")
    with open(latest_file, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("No output file found in outputs/ yet.")

🎯 USER REQUEST: I am going to bangkok tomorrow to attend a business meeting for 2 weeks, so give me what should I pack and plan a refreshment idea after the meeting for the last three days.
------------------------------------------------------------
🔄 [Step 1/6] Reasoning...
🛠️  Action: get_city_activities({"theme": "refreshment", "num_days": 3, "city": "Bangkok"})
👀 Observation: success
🔄 [Step 2/6] Reasoning...
🛠️  Action: get_weather_forecast({"days": 7, "city": "Bangkok"})
👀 Observation: success
🔄 [Step 3/6] Reasoning...
🛠️  Action: save_travel_plan({"content": "# Bangkok Business Trip & Post-Meeting Refreshment Itinerary (2 Weeks)\n\n## Destination Overview\n- **City:** Bangkok, Thailand\n- **Duration:** 2 Weeks (Business + Post-Meeting Refreshment)\n- **Weather Outlook:** Hot and sunny with temperatures averaging around 35\u00b0C (highs) and 26\u00b0C (lows), with potential tropical humidity and brief rain showers.\n\n---\n\n## Part 1: Comprehensive Packing List\n\n### 1. Profes